# signal_state_best_n.pt 파인튜닝 (좌회전 데이터 추가)

기존 `red`/`green_straight`/`green_left` 3클래스 모델을 좌회전 데이터로 이어서 학습한다.

**진행 전 확인:** 새 좌회전 이미지(`0823_좌회전_선택/`, 86장)는 아직 bbox 라벨이 없다 —
Roboflow(또는 CVAT/LabelImg)로 `green_left`(class 2) 박스를 먼저 그리고, 기존
`datasets/signal_state/`에 새 배치로 합쳐서 export한 `data.yaml`을 써야 한다.
새 86장만 단독으로 학습하지 말 것 — 전부 `green_left`뿐이라 `red`/`green_straight`를
모델이 잊어버리는 catastrophic forgetting 위험이 크다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics

## 경로 설정 — 아래 3개를 실제 Drive 경로로 바꿀 것

In [ ]:
DATA_YAML  = '/content/drive/MyDrive/UMK/datasets/signal_state/data.yaml'          # TODO
BASE_CKPT  = '/content/drive/MyDrive/UMK/yolo_ros/signal_state_best_n.pt'          # TODO: 기존 학습된 .pt
ONNX_DEST  = '/content/drive/MyDrive/UMK/yolo_ros/signal_state_best_n.onnx'        # TODO: 최종 onnx 저장 위치

# data.yaml 형식 확인용 예시:
# train: /content/drive/MyDrive/.../signal_state/train/images
# val:   /content/drive/MyDrive/.../signal_state/valid/images
# nc: 3
# names: ['red', 'green_straight', 'green_left']  # config.py YOLO_SIGNAL_STATE_CLASS_NAMES와 순서 반드시 일치

## 이어서 학습 (파인튜닝)

`yolov8n.pt`(COCO 베이스)가 아니라 **이미 학습된 `signal_state_best_n.pt`**를 불러와서
바로 `.train()`을 부르면 그게 곧 이어서 학습(파인튜닝)이다 — `red`/`green_straight`를
이미 아는 가중치에서 시작하므로 그 두 클래스를 새로 배우는 게 아니라 유지한 채
`green_left`만 보강된다.

(`resume=True`는 여기 쓰는 게 아님 — 그건 중단된 학습을 같은 run에서 이어갈 때 쓰는
다른 옵션. 지금처럼 "이미 끝난 모델을 새 데이터로 더 학습"할 땐 그냥 아래처럼 한다.)

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_CKPT)

results = model.train(
    data=DATA_YAML,
    epochs=30,            # 처음 학습보다 짧게 — 이미 수렴한 모델의 미세조정
    imgsz=640,
    batch=16,
    lr0=0.001,             # 기본값(0.01)보다 낮춰 기존 클래스 붕괴 방지
    patience=10,           # val 성능 정체 시 조기종료
    project='signal_state_finetune',
    name='left_boost_0823',
    exist_ok=True,
)

## 검증 — 클래스별 성능 확인

`green_left`만 오르고 `red`/`green_straight`가 떨어졌으면 lr을 더 낮추거나 epoch을
줄여 재시도할 것.

In [ ]:
metrics = model.val(data=DATA_YAML)
print(metrics.box.maps)   # 클래스별 mAP50-95: [red, green_straight, green_left] 순

# runs/detect/.../confusion_matrix.png 도 열어서 green_left가 실제로 green_straight나
# 배경과 헷갈리고 있는지 눈으로 확인할 것

## ONNX export (기존 규약 그대로: imgsz=640, opset=12, simplify=True, nms=True)

In [ ]:
best_pt = model.trainer.best   # runs/.../weights/best.pt
best_model = YOLO(best_pt)
best_model.export(format='onnx', imgsz=640, opset=12, simplify=True, nms=True)

In [ ]:
import shutil
shutil.copy(str(best_pt).replace('.pt', '.onnx'), ONNX_DEST)
print('저장 완료:', ONNX_DEST)

## 마지막

Drive에서 나온 새 `signal_state_best_n.onnx`를 저장소 `yolo_ros/signal_state_best_n.onnx`에
덮어쓰고, 실차에서 `DEBUG_VIZ_YOLO_SIGNAL_STATE=True`로 기존 Hough 판정과 비교 검증할 것.
실차 미검증 상태로는 `perc_signal()` 판단 소스로 바꾸지 않는다(저장소 관례).